# Lab02 钢球识别两种方案对比
姓名：fgssxg
## 实验目的
对比传统高光识别（OpenCV手工特征）与 YOLO 目标检测两种钢球识别方案，理解手工视觉方法和深度学习检测方法各自的特点、优势与局限。

## 方案一：传统高光点识别
### 原理
利用钢球表面反光形成的高光斑点进行识别，基于 OpenCV 实现：
1. 图像转到 HSV 空间，设置颜色阈值筛选高光区域，生成掩码 mask；
2. 使用形态学 kernel 做腐蚀、膨胀，去除噪点、平滑轮廓；
3. 查找轮廓，对每个候选轮廓计算：面积、圆形度（光滑度）、长宽比；
4. 多约束加权得到 score，选出得分最高的轮廓作为钢球，输出圆心坐标。

### 优缺点
✅ 优点：不需要数据集、不需要训练，代码简单、推理速度快，适合固定光照、固定背景的场景。
❌ 缺点：全部是人工设定的硬阈值约束。一旦光线、背景改变，高光mask直接失效，识别准确率暴跌，泛化能力很差。

参考：传统钢球高光跟踪开源方案，原理已在下方代码复现。

In [ ]:
import cv2
import numpy as np

# 读取图片，你可以换成摄像头 cap = cv2.VideoCapture(0)
img = cv2.imread("steelball.jpg")
if img is None:
    print("提示：请把一张钢球图片命名为 steelball.jpg 和本文件放在同一目录")
else:
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # 高光阈值，这里是示例，实际需要调参
    lower_highlight = np.array([0, 0, 220])
    upper_highlight = np.array([180, 30, 255])
    mask = cv2.inRange(hsv, lower_highlight, upper_highlight)

    # 形态学滤波
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if 10 < area < 500:
            perimeter = cv2.arcLength(cnt, True)
            circularity = 4 * np.pi * area / (perimeter * perimeter) if perimeter>0 else 0
            if circularity > 0.7:
                (x,y), r = cv2.minEnclosingCircle(cnt)
                cv2.circle(img, (int(x),int(y)), int(r), (0,255,0), 2)
                print(f"检测到钢球候选：坐标({x:.1f},{y:.1f})，圆形度：{circularity:.2f}")

    cv2.imshow("mask", mask)
    cv2.imshow("result", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

## 方案二：YOLO 目标检测识别钢球
### 原理
属于数据驱动的深度学习方案：
1. 采集钢球在不同光照、背景下的图片，标注钢球位置，制作数据集；
2. 使用 YOLO 模型在数据集上训练，让模型自动学习钢球的视觉特征；
3. 推理阶段：模型输出检测框以及置信度 confidence；
4. 设定置信度阈值，大于阈值就判定为钢球。

### 优缺点
✅ 优点：模型自主学习特征，对光照变化、背景干扰鲁棒性强，泛化能力好。
❌ 缺点：前期需要大量图片标注、训练，工作量大，对算力有一定要求。

参考：YOLO钢球部署与导出方案，原理如下代码示意。

## 两种方案对比总结

| 方案 | 泛化能力 | 前期工作量 | 推理速度 |
| ---- | -------- | ---------- | -------- |
| 传统高光识别 | 差 | 低（调阈值） | 很快 |
| YOLO检测 | 好 | 高（标注+训练） | 中等 |

### 思考
比赛场景如果光照稳定、背景简单，可以用传统高光方案，速度快；
赛场光照多变、背景复杂，优先YOLO，牺牲一点速度换取稳定性。

In [ ]:
# 这里只是示意逻辑，需要提前训练好钢球的YOLO模型
from ultralytics import YOLO

# model = YOLO("steelball_yolo.pt")
# res = model("test.jpg", conf=0.6) # conf就是置信度阈值
# res[0].show()
print("YOLO流程：加载模型 → 输入图像 → 输出检测框+置信度 → 和阈值比较判定目标")